1. 문서의 내용을 읽는다
2. 문서를 쪼갠다
    - 토큰수 초과로 답변을 생성하지 못할 수 있고
    - 문서가 길면 (인풋이 길면) 답변 생성이 오래 걸림
3. 임베딩(텍스트를 숫자 벡터로 변환) -> 벡터 데이터베이스에 저장
4. 질문이 있을 대, 벡터 데이터베이스에 유사도 검색
5. 유사도 검색으로 가져온 문서를 LLM에 질문과 같이 전달달

1. 문서의 내용을 읽는다.

In [ ]:
%pip install -qU  docx2txt langchain-community

In [ ]:
from langchain_community.document_loaders import Docx2txtLoader

loader = Docx2txtLoader('./tax.docx')


2. 문서의 내용을 쪼갠다.

In [ ]:
%pip install -qU langchain-text-splitters

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, TextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500, # 하나의 청크가 가질 수 있는 토큰 수수
    chunk_overlap=200, # 청크간 겹치는 정도, 유사도 검사를 할 때 우너하는 정보를 가져올 확률을 높인다.
)

document_list = loader.load_and_split(text_splitter=text_splitter)

3. 쪼개진 문서를 임베딩한다. 이후 벡터 데이터베이스에 저장한다.

In [ ]:
%pip install langchain-huggingface sentence-transformers

In [ ]:
!ollama pull nomic-embed-text
%pip install langchain-ollama

In [ ]:
%pip install python-dotenv langchain-upstage

In [ ]:
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings

load_dotenv()

upstage_embedding = UpstageEmbeddings(model="solar-embedding-1-large")

In [ ]:
from langchain_ollama import OllamaEmbeddings

ollama_embedding = OllamaEmbeddings(model="nomic-embed-text")

In [ ]:
%pip install langchain-pinecone pinecone-client

In [ ]:
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone
import os

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

database = PineconeVectorStore.from_documents(
    documents=document_list,
    embedding=upstage_embedding,
    index_name="tax-table-index"
)

In [ ]:
query = '연봉 5천만원인 직장인의의 소득세는 얼마인가요?'
retrieved_docs = database.similarity_search(query, k=10)

In [ ]:
retrieved_docs

In [ ]:
from langchain_anthropic import ChatAnthropic

llm = ChatAnthropic(model="claude-haiku-4-5")

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

dictionary = ["사람을 나타내는 표현 -> 거주자자"]

dictionary_prompt = ChatPromptTemplate.from_template(f"""
    사용자의 질문을 보고, 우리의 사전을 참고해서 사용자의 질문을 변경해주세요.
    만약 변경할 필요가 없다고 판단된다면, 사용자의 질문을 변경하지 않아도 됩니다.
    사전: {dictionary}

    질문: {{question}}
    """
    )

newQuery_chain = dictionary_prompt | llm | StrOutputParser()


In [ ]:
new_question = newQuery_chain.invoke({"question": query})

In [ ]:
prompt = f"""[Identity]
- 당신은 최고의 한국 소득세 전문가 입니다.
- [Context]를 참고해서 사용자의 질문에 답변해주세요
- 결과만 출력해주세요력해주세요

[Context]
{retrieved_docs}

Question: {new_question}
"""

In [ ]:
ai_message = llm.invoke(prompt)

In [ ]:
ai_message.content

'# 연봉 5천만원 직장인의 소득세 계산\n\n연봉 5,000만원인 직장인의 소득세를 계산하겠습니다.\n\n## 1단계: 근로소득공제\n\n**총급여액: 5,000만원**\n\n근로소득공제 계산:\n- 5,000만원 × 70% - 540만원 = **2,960만원**\n- (공제액 한도: 2,000만원 이하)\n- **공제액: 2,000만원** ✓\n\n## 2단계: 종합소득금액\n\n- 근로소득금액 = 5,000만원 - 2,000만원 = **3,000만원**\n\n## 3단계: 과세표준\n\n기본공제(본인 기준): 150만원\n\n- **과세표준 = 3,000만원 - 150만원 = 2,850만원**\n\n## 4단계: 산출세액\n\n종합소득세 세율(2850만원 구간): **15%**\n\n- 산출세액 = 2,850만원 × 15% = **427.5만원**\n\n## 5단계: 근로소득세액공제\n\n총급여액 5,000만원은 7,000만원 이하이므로:\n\n- 공제액 = 74만원 - [(5,000만원 - 3,300만원) × 8/1000]\n- = 74만원 - 13.6만원 = **60.4만원** ✓\n\n## 최종 소득세\n\n**427.5만원 - 60.4만원 = 약 367만원**\n\n> 💡 **참고**: 위 계산은 기본공제만 반영한 기준입니다. 실제로는 배우자, 부양가족, 보험료, 의료비, 교육비 등 다양한 추가공제와 특별세액공제가 적용될 수 있으므로 실제 납부 세액은 달라질 수 있습니다.'